# **Movie Recommendation System: Collaborative, Content-Based & Hybrid Filtering**

---

### Project Overview
This project builds a complete **Movie Recommendation System** on the [MovieLens dataset](https://grouplens.org/datasets/movielens/), covering three complementary approaches:

- **Collaborative Filtering** — Five algorithms (SVD, ALS, SVD++, KNNBaseline User-Based, KNNBaseline Item-Based) are benchmarked via 5-fold cross-validation. The best-performing model is tuned using Randomized Search.
- **Content-Based Filtering** — A user genre preference profile is built from each user's rated movies and used to score unseen films via dot product similarity.
- **Hybrid Filtering** — Collaborative and content-based scores are normalized and combined with a weighted average to produce more robust, personalized recommendations.

---

### Workflow

| Step | Description |
|------|-------------|
| **1. Data Preparation** | Merge `ratings.csv` and `movies.csv` into a unified dataset with `userId`, `movieId`, `rating`, `title` |
| **2. Model Benchmarking** | Compare SVD, ALS, SVD++, and KNNBaseline using 5-fold cross-validation |
| **3. Hyperparameter Tuning** | Optimize the best model (KNNBaseline Item-Based) using Randomized Search |
| **4. Rating Prediction** | Predict ratings for selected movies across specific users |
| **5. Collaborative Recommendation** | Generate a personalized top-10 list using predicted ratings |
| **6. Content-Based Recommendation** | Build user genre profile and score unseen movies by genre similarity |
| **7. Hybrid Recommendation** | Combine both scores to produce balanced recommendations |

---

### Target Predictions

**Movies:**

| Title | movieId |
|-------|---------|
| Toy Story (1995) | 1 |
| Black Butler: Book of the Atlantic (2017) | 193581 |
| Flint (2017) | 193585 |
| Andrew Dice Clay: Dice Rules (1991) | 193609 |

**Users:** 3, 20, 50, 600

---

> **Key Question:** How do Collaborative, Content-Based, and Hybrid filtering each approach personalized movie recommendations — and where does each method fall short?

# **Import Library**

In [1]:
import pandas as pd
import numpy as np

from surprise import Reader, Dataset
from surprise import SVD, BaselineOnly, SVDpp, KNNBaseline
from surprise.model_selection import KFold, cross_validate, RandomizedSearchCV

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings("ignore")

# **Load Dataset**

In [3]:
# Load user ratings dataset containing userId, movieId, rating, and timestamp
df_rating = pd.read_csv("dataset/ratings_collab.csv")
df_rating.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
# Load movies dataset containing movieId, title, and genres
df_movies = pd.read_csv("dataset/movies_collab.csv")
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


# **Data Cleaning**

In [50]:
# Summarize each column's dtype, null count, null percentage, and cardinality for both datasets
list_df = {
    "df_rating": df_rating,
    "df_movies": df_movies
}

for df_name, df in list_df.items():
    df_item = []

    for col in df.columns:
        df_item.append([
            col, 
            df[col].dtype, 
            df[col].isna().sum(), 
            round(df[col].isna().sum()/len(df[col])*100, 2), 
            df[col].nunique()
        ])

    df_overview = pd.DataFrame(
        columns=["column_name", "data_type", "null", "null_percentage", "unique"], 
        data=df_item
    )

    print(f"Number of rows and columns in {df_name} : {df.shape}")
    display(df_overview)

Number of rows and columns in df_rating : (100836, 4)


,column_name,data_type,null,null_percentage,unique
0,userId,int64,0,0.0,610
1,movieId,int64,0,0.0,9724
2,rating,float64,0,0.0,10
3,timestamp,int64,0,0.0,85043


Number of rows and columns in df_movies : (9742, 3)


,column_name,data_type,null,null_percentage,unique
0,movieId,int64,0,0.0,9742
1,title,str,0,0.0,9737
2,genres,str,0,0.0,951


## Duplicated Data

In [51]:
# Check for duplicate rows across both datasets
for df_name, df in list_df.items():
    print(f"Number of duplicate rows in {df_name} : {df.duplicated().sum()} ({df.duplicated().sum()/len(df)}%)")

Number of duplicate rows in df_rating : 0 (0.0%)
Number of duplicate rows in df_movies : 0 (0.0%)


**Findings:** Both datasets are clean — no missing values and no duplicate rows were found. No imputation or deduplication is needed before proceeding to preprocessing.

# **Data Preprocessing**

In [5]:
# Retain only the columns needed for modelling and drop the rest
df_rating_used= df_rating[["userId", "movieId", "rating"]]
df_movies_used = df_movies[["movieId", "title"]]
display(df_rating_used.head(), df_movies_used.head())

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


In [6]:
# Left-join movies onto ratings on movieId to attach movie titles
df_merged = pd.merge(df_rating_used, df_movies_used, how="left", on="movieId")
df_merged

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
100831,610,166534,4.0,Split (2017)
100832,610,168248,5.0,John Wick: Chapter Two (2017)
100833,610,168250,5.0,Get Out (2017)
100834,610,168252,5.0,Logan (2017)


# **Collaborative Filtering**

Collaborative filtering generates recommendations based on **user rating patterns** — the idea is that users who agreed on past movies are likely to agree on future ones. This approach requires no knowledge of movie content (genres, descriptions) and instead learns entirely from the collective behavior of all users.

This section benchmarks five algorithms — **SVD**, **ALS**, **SVD++**, **KNNBaseline User-Based**, and **KNNBaseline Item-Based** — selects the best-performing model, tunes its hyperparameters, and uses it to predict ratings and generate personalized recommendations.

## **Build User Item Rating Matrix**

In [8]:
# Build user-item rating matrix with userId as rows and movieId as columns
user_item_rating_matrix = df_merged.pivot_table(values="rating", index="userId", columns="movieId")
user_item_rating_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,2.5,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Preview the rating matrix for target users and movies to verify sparsity
user_item_rating_matrix.loc[[3, 20, 50, 600], [1, 193581, 193585, 193609]]

movieId,1,193581,193585,193609
userId,,,,
3,NaN,NaN,NaN,NaN
20,NaN,NaN,NaN,NaN
50,3.0,NaN,NaN,NaN
600,2.5,NaN,NaN,NaN


**Observation:** Most values in the matrix are NaN, indicating that users only rate a small fraction of all available movies. This high sparsity is a fundamental characteristic of collaborative filtering problems and is the core challenge the model must address.

## **Modelling**

In [10]:
# Explore rating distribution to determine rating scale
df_merged.describe()

,userId,movieId,rating
count,100836.000000,100836.000000,100836.000000
mean,326.127564,19435.295718,3.501557
std,182.618491,35530.987199,1.042529
min,1.000000,1.000000,0.500000
25%,177.000000,1199.000000,3.000000
50%,325.000000,2991.000000,3.500000
75%,477.000000,8122.000000,4.000000
max,610.000000,193609.000000,5.000000


In [7]:
# Define rating scale and load data into Surprise format
reader_data = Reader(rating_scale=(0.5, 5))

recom_data = Dataset.load_from_df(df_merged[["userId", "movieId", "rating"]], reader_data)
recom_data

### Model Benchmarking

In [29]:
# Instantiate all candidate models with default parameters for benchmarking
kf = KFold(n_splits=5, random_state=42, shuffle=True)

algo_svd = SVD(random_state=42)
algo_als = BaselineOnly(bsl_options={"method": "als"})
algo_svdpp = SVDpp(random_state=42)
algo_knn_user = KNNBaseline(sim_options={"name": "pearson_baseline", "user_based": True})
algo_knn_item = KNNBaseline(sim_options={"name": "pearson_baseline", "user_based": False})

In [27]:
# Benchmark all models with 5-fold cross-validation
models = {
    "SVD": algo_svd,
    "ALS": algo_als,
    "SVD++": algo_svdpp,
    "KNNBaseline User-based": algo_knn_user,
    "KNNBaseline Item-based": algo_knn_item
}

results = []
for name, model in models.items():
    cv_result = cross_validate(model, recom_data, measures=["RMSE", "MAE"], cv=kf, verbose=False)
    results.append({
        "Model": name,
        "RMSE Mean": round(cv_result["test_rmse"].mean(), 4),
        "RMSE Std": round(cv_result["test_rmse"].std(), 4),
        "MAE Mean": round(cv_result["test_mae"].mean(), 4)
    })

benchmark_result = pd.DataFrame(results).sort_values("RMSE Mean").reset_index(drop=True)
benchmark_result

Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using al

,Model,RMSE Mean,RMSE Std,MAE Mean
0,KNNBaseline Item-based,0.8532,0.0055,0.6505
1,SVD++,0.8619,0.0064,0.6608
2,ALS,0.8726,0.0069,0.6727
3,SVD,0.8732,0.0077,0.6709
4,KNNBaseline User-based,0.8780,0.0071,0.6679


**Benchmark Results (5-fold Cross-Validation):**

| Model | RMSE Mean | RMSE Std | MAE Mean |
|-------|-----------|----------|----------|
| KNNBaseline Item-Based | 0.8532 | 0.0055 | 0.6505 |
| SVD++ | 0.8619 | 0.0064 | 0.6608 |
| ALS | 0.8726 | 0.0069 | 0.6727 |
| SVD | 0.8732 | 0.0077 | 0.6709 |
| KNNBaseline User-Based | 0.8780 | 0.0071 | 0.6679 |

**KNNBaseline Item-Based** outperforms all other models consistently across all folds with the lowest mean RMSE and standard deviation, confirming it as the most stable and accurate model. **KNNBaseline Item-Based is selected for hyperparameter tuning.**


## **Hyperparameter Tuning**

### KNNBaseline Item-Based with Randomized Search

In [28]:
# Explore a wide parameter space using Randomized Search to find the optimal KNNBaseline parameters
knn_param_rand = {
    "k": np.arange(10, 100, 5),
    "min_k": np.arange(1, 20, 1),
    "sim_options": {
        "name": ["pearson_baseline", "pearson", "cosine"],
        "user_based": [False]
    }
}

rand_search = RandomizedSearchCV(KNNBaseline, knn_param_rand, measures=["rmse", "mae"], cv=5, n_iter=10, random_state=42)
rand_search.fit(recom_data)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline si

In [30]:
# Display the best score and corresponding parameters found by Randomized Search
print("=" * 35)
print("BEST RESULTS - RANDOMIZED SEARCH")
print("=" * 35)

print("\nRMSE")
print(f"  Best Score : {rand_search.best_score['rmse']:.4f}")
print(f"  Best Params: {rand_search.best_params['rmse']}")

print("\nMAE")
print(f"  Best Score : {rand_search.best_score['mae']:.4f}")
print(f"  Best Params: {rand_search.best_params['mae']}")

BEST RESULTS - RANDOMIZED SEARCH

RMSE
  Best Score : 0.8497
  Best Params: {'k': np.int64(45), 'min_k': np.int64(9), 'sim_options': {'name': 'pearson_baseline', 'user_based': False}}

MAE
  Best Score : 0.6491
  Best Params: {'k': np.int64(45), 'min_k': np.int64(9), 'sim_options': {'name': 'pearson_baseline', 'user_based': False}}


**Randomized Search Results:**
- Best RMSE: **0.8497**
- Best parameters: `k=45`, `min_k=9`, `sim_options={'name': 'pearson_baseline', 'user_based': False}`

Randomized Search explored a wide parameter space across `k`, `min_k`, and similarity metrics. The optimal configuration uses **Pearson Baseline** similarity with **45 neighbors** and a minimum threshold of **9**, achieving a notable improvement over the default KNNBaseline Item-Based RMSE from benchmarking.

### Model Performance Before vs After Tuning

In [31]:
# Cross-validate KNNBaseline Item-Based using the best parameters found by Randomized Search
algo_knn_item_tuned = KNNBaseline(
    k=45,
    min_k=9,
    sim_options={"name": "pearson_baseline", "user_based": False}
)
cv_knn_item_tuned = cross_validate(algo_knn_item_tuned, recom_data, measures=["rmse", "mae"], cv=kf, verbose=True)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBaseline on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8566  0.8452  0.8421  0.8477  0.8567  0.8497  0.0060  
MAE (testset)     0.6518  0.6468  0.6443  0.6487  0.6538  0.6491  0.0034  
Fit time          8.11    8.65    8.20    8.07    8.03    8.21    0.23    
Test time         6.55    6.56    6.19    6.68    5.9

In [32]:
# Compare RMSE before and after hyperparameter tuning
before = benchmark_result.iloc[0, 1]
after = cv_knn_item_tuned["test_rmse"].mean()

improvement = ((before - after) / before) * 100

comparison = pd.DataFrame({
    "Metric": ["RMSE"],
    "Before Tuning": [before],
    "After Tuning": [after],
    "Improvement (%)": [improvement]
})

comparison.round(4)

,Metric,Before Tuning,After Tuning,Improvement (%)
0,RMSE,0.8532,0.8497,0.4148


**Hyperparameter Tuning Summary:**

| Model | CV Mean RMSE |
|-------|-------------|
| KNNBaseline Item-Based (default) | 0.8532 |
| KNNBaseline Item-Based (tuned)   | 0.8497 |

Tuning improved the model's RMSE by reducing prediction error on unseen ratings. The tuned model with `k=45`, `min_k=9`, and Pearson Baseline similarity is used for all final predictions.

## **Prediction Results**

The following movies and users are used to evaluate the final model's prediction capability:

**Movies:**

| Title | movieId |
|-------|---------|
| Toy Story (1995) | 1 |
| Black Butler: Book of the Atlantic (2017) | 193581 |
| Flint (2017) | 193585 |
| Andrew Dice Clay: Dice Rules (1991) | 193609 |

**Users:** 3, 20, 50, 600

### Build User-Movie Prediction DataFrame

In [33]:
# Build prediction dataframe for each user-movie combination
list_users = [3, 20, 50, 600]
list_movies = [1, 193581, 193585, 193609]

list_titles = []
for movie_id in list_movies:
    list_titles.append(df_merged.loc[df_merged["movieId"] == movie_id, "title"].values[0])

df_predict = []
for user_id in list_users:
    for movie_id, title in zip(list_movies, list_titles):
        df_predict.append({"userId": user_id, "movieId": movie_id, "title": title})

df_predict = pd.DataFrame(df_predict)
df_predict

,userId,movieId,title
0,3,1,Toy Story (1995)
1,3,193581,Black Butler: Book of the Atlantic (2017)
2,3,193585,Flint (2017)
3,3,193609,Andrew Dice Clay: Dice Rules (1991)
4,20,1,Toy Story (1995)
5,20,193581,Black Butler: Book of the Atlantic (2017)
6,20,193585,Flint (2017)
7,20,193609,Andrew Dice Clay: Dice Rules (1991)
8,50,1,Toy Story (1995)
9,50,193581,Black Butler: Book of the Atlantic (2017)


### Train Final Model

In [8]:
# Retrain KNNBaseline Item-Based with best hyperparameters on the full dataset for final predictions
best_knn_item = KNNBaseline(
    k=45,
    min_k=9,
    sim_options={"name": "pearson_baseline", "user_based": False}
)
best_knn_item.fit(recom_data.build_full_trainset())

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


### Save Final Model

In [9]:
from operator import truediv
import pickle
import os

# Save trained model for Streamlit app
os.makedirs("models", exist_ok=True)
with open("models/best_knn_item.pkl", "wb") as f:
    pickle.dump(best_knn_item, f)

print("Model saved to models/best_knn_item.pkl")

Model saved to models/best_knn_item.pkl


### Predict Rating for Each Movie

In [40]:
# Generate predicted rating for each user-movie pair — est[3] extracts the estimated score from the prediction tuple
rating_pred = []
for index, row in df_predict.iterrows():
    est = best_knn_item.predict(row["userId"], row["movieId"])
    rating_pred.append(est[3])

df_predict["predict_rating"] = rating_pred

df_predict.sort_values(by=["userId", "predict_rating"], ascending=[True, False], inplace=True)
df_predict.reset_index(drop=True)

prediction_summary = (df_predict.groupby("userId")["predict_rating"].agg(["min", "max"]).round(4))

print("=" * 35)
print("MOVIE RATING PREDICTIONS")
print("=" * 35)

display(df_predict)

print("=" * 35)
print("PREDICTED RATING SUMMARY BY USER")
print("=" * 35)

display(prediction_summary)

MOVIE RATING PREDICTIONS


,userId,movieId,title,predict_rating
3,3,193609,Andrew Dice Clay: Dice Rules (1991),2.717563
1,3,193581,Black Butler: Book of the Atlantic (2017),2.703269
2,3,193585,Flint (2017),2.657814
0,3,1,Toy Story (1995),1.830564
4,20,1,Toy Story (1995),4.563666
7,20,193609,Andrew Dice Clay: Dice Rules (1991),3.621693
5,20,193581,Black Butler: Book of the Atlantic (2017),3.607400
6,20,193585,Flint (2017),3.561945
8,50,1,Toy Story (1995),2.940205
11,50,193609,Andrew Dice Clay: Dice Rules (1991),2.688716


PREDICTED RATING SUMMARY BY USER


,min,max
userId,,
3,1.8306,2.7176
20,3.5619,4.5637
50,2.6290,2.9402
600,2.9400,3.1266


**Prediction Results Interpretation:**

The predicted ratings are driven by **item-item similarity** — KNNBaseline Item-Based identifies movies that are consistently rated in a similar pattern by the same group of users:

| User | Predicted Range | Interpretation |
|------|----------------|----------------|
| User 3   | 1.83 – 2.72 | Lowest overall predictor — gives the lowest scores among all four users |
| User 20  | 3.56 – 4.56 | Optimistic rater — consistently gives high scores across all movies |
| User 50  | 2.63 – 2.94 | Conservative rater — moderate-low scores, slightly above User 3 |
| User 600 | 2.94 – 3.13 | Moderate rater — scores close to the global average |

Unlike simpler models, **KNNBaseline Item-Based** captures individual rating tendencies — Toy Story (1995) receives a high prediction for User 20 (4.56) but a very low one for User 3 (1.83), reflecting each user's unique viewing profile rather than a global popularity bias.


## **Movie Recommendations per User**

In [37]:
# Display watch history for demo user (userId 409) to understand their rating taste
df_merged[df_merged["userId"] == 409]

,userId,movieId,rating,title
61716,409,39,4.0,Clueless (1995)
61717,409,111,5.0,Taxi Driver (1976)
61718,409,223,4.0,Clerks (1994)
61719,409,235,5.0,Ed Wood (1994)
61720,409,260,4.0,Star Wars: Episode IV - A New Hope (1977)
...,...,...,...,...
61837,409,3836,5.0,Kelly's Heroes (1970)
61838,409,3846,5.0,Easy Money (1983)
61839,409,3868,4.0,"Naked Gun: From the Files of Police Squad!, Th..."
61840,409,3869,4.0,"Naked Gun 2 1/2: The Smell of Fear, The (1991)"


**Watch History Overview (userId 409):**

userId 409 has rated **126 movies**, predominantly giving scores of **4.0–5.0**. Their watch history spans cult comedies (*Clerks*, *Ed Wood*), crime dramas (*Taxi Driver*), action-adventure (*Star Wars: Episode IV*), and classic comedies (*Naked Gun*) — indicating a user with eclectic but quality-conscious taste.

In [105]:
# Score all unrated movies for demo user and extract top-10 by predicted rating
user_id = 409
watched_movieIds = df_merged[df_merged["userId"] == user_id]["movieId"].tolist()

all_movieIds = list(df_merged.drop_duplicates("movieId")["movieId"])
all_movieIds_unseen = [mid for mid in all_movieIds if mid not in watched_movieIds]

all_title = df_merged.drop_duplicates("movieId").set_index("movieId")["title"]

# Identify titles that appear more than once across all movies (duplicate titles with different movieId)
title_counts = all_title.value_counts()

def disambiguate(movie_id, title):
    if title_counts[title] > 1:
        return f"{title} [id:{movie_id}]"
    return title

movie_score_pred = [best_knn_item.predict(user_id, movie_id).est for movie_id in all_movieIds_unseen]

df_recom_collab = pd.DataFrame({
    "movieId" : all_movieIds_unseen,
    "title" : [disambiguate(mid, all_title[mid]) for mid in all_movieIds_unseen],
    "predict_score": movie_score_pred
}).sort_values(by="predict_score", ascending=False)

df_recom_collab.head(10).reset_index(drop=True)

,movieId,title,predict_score
0,750,Dr. Strangelove or: How I Learned to Stop Worr...,4.719852
1,1204,Lawrence of Arabia (1962),4.687672
2,58301,Funny Games U.S. (2007),4.660654
3,5618,Spirited Away (Sen to Chihiro no kamikakushi) ...,4.600773
4,1261,Evil Dead II (Dead by Dawn) (1987),4.599212
5,1104,"Streetcar Named Desire, A (1951)",4.598972
6,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.594922
7,933,To Catch a Thief (1955),4.562627
8,1233,"Boot, Das (Boat, The) (1981)",4.541364
9,1262,"Great Escape, The (1963)",4.540241


**Collaborative Recommendation Results (userId 409):**

The top-10 recommended movies span classic Hollywood cinema, cult favorites, and world cinema (e.g., *Dr. Strangelove*, *Lawrence of Arabia*, *Funny Games U.S.*, *Evil Dead II*, *Spirited Away*). This aligns with **KNNBaseline Item-Based** behavior — it identifies movies that share similar rating patterns with films this user has already rated, naturally converging toward critically regarded and cult titles across multiple genres. Given this user's 126 ratings skewed toward 4.0–5.0 scores, the model surfaces films consistently rated highly by users with similar viewing patterns.

# **Content-Based Filtering**

Content-based filtering recommends movies based on **genre similarity** to a user's watch history. Unlike collaborative filtering which relies on other users' behavior, this approach builds a personal genre profile from the user's own rated movies.

Each genre is **weighted by the user's actual rating** — movies the user rated highly contribute more to their preference profile. The profile is then used to score all unseen movies via dot product, returning the most genre-relevant recommendations.

## **Build Genre Feature Matrix**

In [99]:
# Exclude movies with no genre information
df_movies_genre = df_movies[df_movies["genres"] != "(no genres listed)"].copy().reset_index(drop=True)

# Convert genre strings into binary vectors using CountVectorizer with a pipe-separated tokenization
vect = CountVectorizer(tokenizer=lambda x: x.split("|"), token_pattern=None)
df_genre_matrix = vect.fit_transform(df_movies_genre["genres"])
genre_cols = vect.get_feature_names_out()

df_genre_matrix = pd.DataFrame(
    df_genre_matrix.toarray(),
    columns=genre_cols
)
df_genre_matrix = pd.concat([df_movies_genre[["movieId", "title"]], df_genre_matrix], axis=1)

print(f"Genre matrix shape: {df_genre_matrix.shape}")
print(f"Total genres: {len(genre_cols)}")
df_genre_matrix.head()

Genre matrix shape: (9708, 21)
Total genres: 19


,movieId,title,action,adventure,animation,children,comedy,crime,documentary,drama,...,film-noir,horror,imax,musical,mystery,romance,sci-fi,thriller,war,western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Genre Matrix Overview:**

The genre matrix contains **9,708 movies** across **19 unique genres**. Each row represents a movie and each column a genre — values are binary (1 = genre present, 0 = absent). This matrix is the foundation for computing genre-based similarity between movies.

## **User Genre Profile**

In [103]:
# Build genre preference profile for demo user weighted by their own ratings
user_id = 409

df_user_watched_movies = df_merged[df_merged["userId"] == user_id][["movieId", "title", "rating"]]
watched_movieIds = df_user_watched_movies["movieId"].tolist()

df_genre_watched = df_genre_matrix[df_genre_matrix["movieId"].isin(watched_movieIds)].copy()
df_genre_watched = df_genre_watched.merge(df_user_watched_movies[["movieId", "rating"]], on="movieId")

# Weight each genre by user rating — higher-rated movies contribute more to the profile
df_weighted_genres = df_genre_watched.loc[:, "action":"western"].multiply(df_genre_watched["rating"].values, axis=0)

# Normalize to probability distribution summing to 1
user_profile = df_weighted_genres.sum() / df_weighted_genres.sum().sum()

df_user_profile = pd.DataFrame({
    "genre": user_profile.index,
    "weight": (user_profile.values * 100).round(2)
}).sort_values("weight", ascending=False).reset_index(drop=True)

df_user_profile

,genre,weight
0,comedy,31.57
1,drama,20.91
2,crime,8.63
3,romance,7.61
4,thriller,5.38
5,action,4.67
6,adventure,3.76
7,horror,3.35
8,fantasy,3.05
9,sci-fi,2.54


**User 409 Genre Profile:**

| Genre | Weight |
|-------|--------|
| Comedy | 31.57% |
| Drama | 20.91% |
| Crime | 8.63% |
| Romance | 7.61% |
| Thriller | 5.38% |

userId 409 has a strong preference toward **Comedy** and **Drama**, which together account for over 52% of their weighted genre profile. Genres with 0% weight (Animation, Film-Noir, IMAX) are absent from their watch history and contribute nothing to scoring.

## **Content-Based Recommendations**

In [98]:
# Score all unwatched movies using dot product between genre vectors and user profile
df_user_unwatched_movies = df_genre_matrix[~df_genre_matrix["movieId"].isin(watched_movieIds)].copy().reset_index(drop=True)

content_scores = df_user_unwatched_movies.loc[:, "action":"western"].multiply(user_profile.values, axis=1)

df_recom_content = pd.DataFrame({
    "movieId": df_user_unwatched_movies["movieId"].values,
    "title": df_user_unwatched_movies["title"].values,
    "content_score": content_scores.sum(axis=1)
}).sort_values("content_score", ascending=False).reset_index(drop=True)

df_recom_content.head(10)

,movieId,title,content_score
0,81132,Rubber (2010),0.808122
1,4719,Osmosis Jones (2001),0.787817
2,7235,Ichi the Killer (Koroshiya 1) (2001),0.745178
3,1912,Out of Sight (1998),0.741117
4,3893,Nurse Betty (2000),0.741117
5,144606,Confessions of a Dangerous Mind (2002),0.741117
6,4956,"Stunt Man, The (1980)",0.739086
7,970,Beat the Devil (1953),0.724873
8,7835,Song of the Thin Man (1947),0.718782
9,31921,"Seven-Per-Cent Solution, The (1976)",0.712690


**Content-Based Recommendation Results (userId 409):**

The top results are driven by genre overlap with userId 409's profile — movies scoring highest are those combining Comedy, Drama, and Crime, which together make up ~61% of the user's genre preference. Unlike collaborative filtering, these results depend entirely on genre composition: a movie with a perfect genre match receives a high score regardless of its popularity or global rating.

Note that content-based filtering has no notion of rating quality — a niche film with the right genres ranks equally with a critically acclaimed one.

# **Hybrid Filtering**

Hybrid filtering combines both approaches to compensate for their individual weaknesses:

- **Collaborative filtering** captures user taste from rating patterns but struggles with niche or less-rated movies (*cold start*)
- **Content-based filtering** covers niche movies through genre matching but ignores actual rating quality

By blending both scores with a weighted average, the hybrid model balances **personalization** (collaborative) with **genre relevance** (content-based).

## **Combine Scores**

In [107]:
# Merge collaborative and content-based scores on movieId
alpha = 0.6  # weight for collaborative; (1 - alpha) for content-based

df_recom_hybrid = df_recom_collab[["movieId", "title", "predict_score"]].merge(
    df_recom_content[["movieId", "content_score"]],
    on="movieId",
    how="inner"
)

# Normalize both scores to [0, 1] — required since they have different scales
scaler = MinMaxScaler()
df_recom_hybrid[["predict_score_norm", "content_score_norm"]] = scaler.fit_transform(df_recom_hybrid[["predict_score", "content_score"]])

# Combine: hybrid = α × collaborative + (1 - α) × content-based
df_recom_hybrid["hybrid_score"] = (alpha * df_recom_hybrid["predict_score_norm"] + (1 - alpha) * df_recom_hybrid["content_score_norm"])

df_recom_hybrid = df_recom_hybrid.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
df_recom_hybrid[["title", "predict_score", "content_score", "hybrid_score"]].head(10)

,title,predict_score,content_score,hybrid_score
0,"Stunt Man, The (1980)",4.275742,0.739086,0.861810
1,"Philadelphia Story, The (1940)",4.521247,0.601015,0.850970
2,Man Bites Dog (C'est arrivé près de chez vous)...,4.346913,0.664975,0.841796
3,In Bruges (2008),4.330184,0.664975,0.837878
4,My Fair Lady (1964),4.411811,0.622335,0.835891
5,Forrest Gump (1994),4.345838,0.625381,0.821947
6,Adaptation (2002),4.391924,0.601015,0.820680
7,"Blind Swordsman: Zatoichi, The (Zatôichi) (2003)",4.263652,0.657868,0.818778
8,Harold and Maude (1971),4.368270,0.601015,0.815140
9,Life Is Beautiful (La Vita è bella) (1997),4.294021,0.625381,0.809810


**Hybrid Recommendation Results (userId 409):**

| Title | Collaborative | Content | Hybrid |
|-------|--------------|---------|--------|
| Stunt Man, The (1980) | 4.28 | 0.74 | 0.862 |
| Philadelphia Story, The (1940) | 4.52 | 0.60 | 0.851 |
| Man Bites Dog (1992) | 4.35 | 0.66 | 0.842 |
| In Bruges (2008) | 4.33 | 0.66 | 0.838 |
| My Fair Lady (1964) | 4.41 | 0.62 | 0.836 |

The hybrid model surfaces movies that score well on **both** dimensions — high predicted ratings from collaborative filtering and strong genre alignment with the user's Comedy/Drama-heavy profile. Films like *Forrest Gump* and *Philadelphia Story* rise to the top because they combine strong rating signals with genre relevance.

Classic films that ranked highly in collaborative filtering alone (e.g., *Dr. Strangelove*, *Lawrence of Arabia*) may rank lower here if their genre composition (War, History) diverges from the user's primary preferences — demonstrating the complementary nature of the two approaches.

# **Conclusions**

## Method Comparison

| Method | Basis | Strength | Weakness |
|--------|-------|----------|----------|
| **Collaborative Filtering** | User rating patterns | Captures real taste | Needs rating history |
| **Content-Based Filtering** | Genre similarity | Works for niche films | Ignores rating quality |
| **Hybrid** | Weighted combination | Balances both | Requires both inputs |

## Model Performance

| Stage | Model | RMSE |
|-------|-------|------|
| Benchmarking (best) | KNNBaseline Item-Based | 0.8532 |
| After tuning (k=45, min_k=9) | KNNBaseline Item-Based | 0.8497 |
| Improvement | — | 0.41% |

## Key Findings

- **KNNBaseline Item-Based** outperformed SVD, SVD++, ALS, and KNNBaseline User-Based across all 5-fold cross-validation folds
- Hyperparameter tuning via **RandomizedSearchCV** improved RMSE from 0.8532 → 0.8497
- The hybrid model with `alpha = 0.6` balances collaborative quality signals with content-based genre relevance, producing more diverse and defensible recommendations than either method alone